In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

In [2]:
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

OUTPUT_DIR = Path("mmi711_outputs_multilabel")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Original event-presence dataset
data = np.load(OUTPUT_DIR / "mmi711_multilabel_variable_length_datasets.npz")

# New sequence-location labels
loc_data = np.load(OUTPUT_DIR / "mmi711_sequence_location_labels.npz")

with open(OUTPUT_DIR / "dataset_config_multilabel_variable_lengths.json", "r") as f:
    config = json.load(f)

LENGTHS = config["lengths"]
EVENT_LABELS = config["event_labels"]

print("Lengths:", LENGTHS)
print("Event labels:", EVENT_LABELS)
print("Location label arrays:", loc_data.files[:5], "...")

Device: cuda
Lengths: [400, 800, 1200]
Event labels: ['mean_shift', 'variance_shift', 'trend_shift', 'point_anomaly', 'collective_anomaly']
Location label arrays: ['Yloc_train_L400', 'Yloc_train_L800', 'Yloc_train_L1200', 'Yloc_val_L400', 'Yloc_val_L800'] ...


In [3]:
X_train_by_length = {}
Yloc_train_by_length = {}

X_val_by_length = {}
Yloc_val_by_length = {}

X_ood_params_by_length = {}
Yloc_ood_params_by_length = {}

X_ood_background_by_length = {}
Yloc_ood_background_by_length = {}

for length in LENGTHS:
    X_train_by_length[length] = data[f"X_train_L{length}"]
    Yloc_train_by_length[length] = loc_data[f"Yloc_train_L{length}"]

    X_val_by_length[length] = data[f"X_val_L{length}"]
    Yloc_val_by_length[length] = loc_data[f"Yloc_val_L{length}"]

    X_ood_params_by_length[length] = data[f"X_ood_params_L{length}"]
    Yloc_ood_params_by_length[length] = loc_data[f"Yloc_ood_params_L{length}"]

    X_ood_background_by_length[length] = data[f"X_ood_background_L{length}"]
    Yloc_ood_background_by_length[length] = loc_data[f"Yloc_ood_background_L{length}"]

    print(
        f"Length {length}:",
        "train", X_train_by_length[length].shape, Yloc_train_by_length[length].shape,
        "| val", X_val_by_length[length].shape, Yloc_val_by_length[length].shape,
        "| ood_params", X_ood_params_by_length[length].shape, Yloc_ood_params_by_length[length].shape,
        "| ood_background", X_ood_background_by_length[length].shape, Yloc_ood_background_by_length[length].shape
    )

Length 400: train (1600, 400) (1600, 400, 5) | val (320, 400) (320, 400, 5) | ood_params (320, 400) (320, 400, 5) | ood_background (320, 400) (320, 400, 5)
Length 800: train (1600, 800) (1600, 800, 5) | val (320, 800) (320, 800, 5) | ood_params (320, 800) (320, 800, 5) | ood_background (320, 800) (320, 800, 5)
Length 1200: train (1600, 1200) (1600, 1200, 5) | val (320, 1200) (320, 1200, 5) | ood_params (320, 1200) (320, 1200, 5) | ood_background (320, 1200) (320, 1200, 5)


In [4]:
def check_location_labels(Yloc_by_length, split_name):
    print("\n" + "=" * 70)
    print(split_name)
    print("=" * 70)

    Y_all = np.concatenate(
        [Yloc_by_length[length].reshape(-1, len(EVENT_LABELS)) for length in LENGTHS],
        axis=0
    )

    for i, label in enumerate(EVENT_LABELS):
        values, counts = np.unique(Y_all[:, i], return_counts=True)
        print(label, dict(zip(values.tolist(), counts.tolist())))

check_location_labels(Yloc_train_by_length, "Train")
check_location_labels(Yloc_val_by_length, "Validation")


Train
mean_shift {0: 3181349, 1: 565584, 2: 93067}
variance_shift {0: 3181061, 1: 563623, 2: 95316}
trend_shift {0: 3189003, 1: 549941, 2: 101056}
point_anomaly {0: 3839277, 1: 723}
collective_anomaly {0: 3730309, 1: 109691}

Validation
mean_shift {0: 635237, 1: 111764, 2: 20999}
variance_shift {0: 629135, 1: 115898, 2: 22967}
trend_shift {0: 637044, 1: 111888, 2: 19068}
point_anomaly {0: 767848, 1: 152}
collective_anomaly {0: 745741, 1: 22259}


In [5]:
class TimeSeriesLocationDataset(Dataset):
    def __init__(self, X, Yloc):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Yloc = torch.tensor(Yloc, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Yloc[idx]

In [6]:
class CNNBiLSTMLocationModel(nn.Module):
    def __init__(self, num_shift_types=3, num_anomaly_types=2, shift_classes=3, hidden_size=64, dropout=0.2):
        super().__init__()

        self.num_shift_types = num_shift_types
        self.num_anomaly_types = num_anomaly_types
        self.shift_classes = shift_classes

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.bilstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        feature_dim = hidden_size * 2

        self.dropout = nn.Dropout(dropout)

        # For mean/variance/trend shift:
        # output shape will be (batch, length, 3 shift types, shift_classes)
        self.shift_head = nn.Linear(feature_dim, num_shift_types * shift_classes)

        # For point/collective anomaly:
        # output shape will be (batch, length, 2 anomaly types)
        self.anomaly_head = nn.Linear(feature_dim, num_anomaly_types)

    def forward(self, x):
        # x: (batch, length)

        x = x.unsqueeze(1)              # (batch, 1, length)
        conv_features = self.cnn(x)     # (batch, 64, length)
        conv_features = conv_features.transpose(1, 2)  # (batch, length, 64)

        lstm_out, _ = self.bilstm(conv_features)       # (batch, length, hidden*2)
        features = self.dropout(lstm_out)

        shift_logits = self.shift_head(features)
        shift_logits = shift_logits.view(
            x.size(0),
            features.size(1),
            self.num_shift_types,
            self.shift_classes
        )

        anomaly_logits = self.anomaly_head(features)

        return shift_logits, anomaly_logits

In [7]:
BATCH_SIZE = 32

def create_location_loaders(X_by_length, Yloc_by_length, shuffle):
    loaders = []

    for length in LENGTHS:
        loader = DataLoader(
            TimeSeriesLocationDataset(
                X_by_length[length],
                Yloc_by_length[length]
            ),
            batch_size=BATCH_SIZE,
            shuffle=shuffle
        )
        loaders.append(loader)

    return loaders

train_loaders = create_location_loaders(X_train_by_length, Yloc_train_by_length, shuffle=True)
val_loaders = create_location_loaders(X_val_by_length, Yloc_val_by_length, shuffle=False)
ood_params_loaders = create_location_loaders(X_ood_params_by_length, Yloc_ood_params_by_length, shuffle=False)
ood_background_loaders = create_location_loaders(X_ood_background_by_length, Yloc_ood_background_by_length, shuffle=False)

print("Number of train loaders:", len(train_loaders))

Number of train loaders: 3


In [8]:
max_shift_label = 0

for length in LENGTHS:
    max_shift_label = max(
        max_shift_label,
        int(Yloc_train_by_length[length][:, :, :3].max()),
        int(Yloc_val_by_length[length][:, :, :3].max())
    )

SHIFT_CLASSES = max_shift_label + 1

print("Max shift label:", max_shift_label)
print("Number of shift classes:", SHIFT_CLASSES)

Max shift label: 2
Number of shift classes: 3


In [9]:
def compute_anomaly_pos_weight(Yloc_by_length):
    all_anom = []

    for length in LENGTHS:
        # point_anomaly and collective_anomaly columns
        all_anom.append(Yloc_by_length[length][:, :, 3:5].reshape(-1, 2))

    all_anom = np.concatenate(all_anom, axis=0)

    pos = all_anom.sum(axis=0)
    neg = len(all_anom) - pos

    pos_weight = neg / (pos + 1e-8)
    pos_weight = np.clip(pos_weight, 1.0, 50.0)

    return torch.tensor(pos_weight, dtype=torch.float32)


anomaly_pos_weight = compute_anomaly_pos_weight(Yloc_train_by_length).to(device)
print("Anomaly pos_weight:", anomaly_pos_weight)

Anomaly pos_weight: tensor([50.0000, 34.0074], device='cuda:0')


In [10]:
def compute_location_loss(
    shift_logits,
    anomaly_logits,
    Yloc,
    anomaly_pos_weight,
    shift_loss_weight=1.0,
    anomaly_loss_weight=1.0
):
    """
    shift_logits:   (batch, length, 3, shift_classes)
    anomaly_logits: (batch, length, 2)
    Yloc:           (batch, length, 5)
    """

    shift_targets = Yloc[:, :, :3].long()
    anomaly_targets = Yloc[:, :, 3:5].float()

    shift_losses = []

    for shift_idx in range(3):
        logits_i = shift_logits[:, :, shift_idx, :]      # (batch, length, classes)
        target_i = shift_targets[:, :, shift_idx]        # (batch, length)

        loss_i = F.cross_entropy(
            logits_i.reshape(-1, logits_i.size(-1)),
            target_i.reshape(-1)
        )
        shift_losses.append(loss_i)

    shift_loss = torch.stack(shift_losses).mean()

    anomaly_loss = F.binary_cross_entropy_with_logits(
        anomaly_logits,
        anomaly_targets,
        pos_weight=anomaly_pos_weight
    )

    total_loss = shift_loss_weight * shift_loss + anomaly_loss_weight * anomaly_loss

    return total_loss, shift_loss.detach(), anomaly_loss.detach()

In [11]:
def apply_location_predictions(shift_logits, anomaly_logits, anomaly_threshold=0.5):
    shift_pred = torch.argmax(shift_logits, dim=-1)  # (batch, length, 3)

    anomaly_prob = torch.sigmoid(anomaly_logits)

    if np.isscalar(anomaly_threshold):
        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()
    else:
        anomaly_threshold = torch.tensor(
            anomaly_threshold,
            dtype=anomaly_prob.dtype,
            device=anomaly_prob.device
        ).view(1, 1, -1)

        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()

    pred_loc = torch.cat([shift_pred, anomaly_pred], dim=-1)

    return pred_loc, anomaly_prob


def compute_pointwise_event_metrics_from_flat(y_true_flat, y_pred_flat):
    """
    y_true_flat and y_pred_flat shape:
        (total_time_points, 5)

    Shift columns may contain regime labels 0,1,2,...
    For pointwise event detection metrics, shifts are converted to binary:
        active shift = label > 0
    """
    y_true_binary = y_true_flat.copy()
    y_pred_binary = y_pred_flat.copy()

    # Shift columns: active if regime label > 0
    y_true_binary[:, :3] = (y_true_binary[:, :3] > 0).astype(int)
    y_pred_binary[:, :3] = (y_pred_binary[:, :3] > 0).astype(int)

    return {
        "pointwise_macro_f1": f1_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_micro_f1": f1_score(
            y_true_binary,
            y_pred_binary,
            average="micro",
            zero_division=0
        ),
        "pointwise_macro_precision": precision_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_macro_recall": recall_score(
            y_true_binary,
            y_pred_binary,
            average="macro",
            zero_division=0
        ),
        "pointwise_hamming_loss": hamming_loss(
            y_true_binary,
            y_pred_binary
        )
    }


def compute_shift_regime_accuracy_from_flat(y_true_flat, y_pred_flat):
    rows = {}

    for i, label in enumerate(EVENT_LABELS[:3]):
        rows[f"{label}_regime_accuracy"] = accuracy_score(
            y_true_flat[:, i],
            y_pred_flat[:, i]
        )

    return rows



In [12]:
def run_one_epoch_location(model, loaders, optimizer=None, anomaly_threshold=0.5):
    is_train = optimizer is not None

    if is_train:
        model.train()
        loaders = loaders.copy()
        random.shuffle(loaders)
    else:
        model.eval()

    total_loss = 0.0
    total_shift_loss = 0.0
    total_anomaly_loss = 0.0
    total_samples = 0

    all_true_flat = []
    all_pred_flat = []

    for loader in loaders:
        for X_batch, Yloc_batch in loader:
            X_batch = X_batch.to(device)
            Yloc_batch = Yloc_batch.to(device)

            if is_train:
                optimizer.zero_grad()

            with torch.set_grad_enabled(is_train):
                shift_logits, anomaly_logits = model(X_batch)

                loss, shift_loss, anomaly_loss = compute_location_loss(
                    shift_logits=shift_logits,
                    anomaly_logits=anomaly_logits,
                    Yloc=Yloc_batch,
                    anomaly_pos_weight=anomaly_pos_weight
                )

                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            batch_size = X_batch.size(0)

            total_loss += loss.item() * batch_size
            total_shift_loss += shift_loss.item() * batch_size
            total_anomaly_loss += anomaly_loss.item() * batch_size
            total_samples += batch_size

            pred_loc, _ = apply_location_predictions(
                shift_logits,
                anomaly_logits,
                anomaly_threshold=anomaly_threshold
            )

            # Flatten each batch from (batch, length, 5) to (batch*length, 5)
            # This avoids errors when different loaders have different lengths.
            true_flat = Yloc_batch.detach().cpu().numpy().reshape(-1, len(EVENT_LABELS))
            pred_flat = pred_loc.detach().cpu().numpy().reshape(-1, len(EVENT_LABELS))

            all_true_flat.append(true_flat)
            all_pred_flat.append(pred_flat)

    y_true_flat = np.concatenate(all_true_flat, axis=0)
    y_pred_flat = np.concatenate(all_pred_flat, axis=0)

    metrics = compute_pointwise_event_metrics_from_flat(
        y_true_flat,
        y_pred_flat
    )

    metrics.update(
        compute_shift_regime_accuracy_from_flat(
            y_true_flat,
            y_pred_flat
        )
    )

    return {
        "loss": total_loss / total_samples,
        "shift_loss": total_shift_loss / total_samples,
        "anomaly_loss": total_anomaly_loss / total_samples,
        **metrics
    }

In [13]:
model = CNNBiLSTMLocationModel(
    shift_classes=SHIFT_CLASSES,
    hidden_size=64,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",      # we monitor validation pointwise macro F1
    factor=0.5,      # reduce LR by half
    patience=3,      # wait 3 epochs without improvement before reducing LR
    min_lr=1e-6
)

EPOCHS = 300
ANOMALY_THRESHOLD = 0.5

early_stopping_patience = 10
min_delta = 1e-4

history = []

best_val_f1 = -np.inf
epochs_without_improvement = 0

best_model_path = OUTPUT_DIR / "best_location_cnn_bilstm_model.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch_location(
        model,
        train_loaders,
        optimizer=optimizer,
        anomaly_threshold=ANOMALY_THRESHOLD
    )

    val_metrics = run_one_epoch_location(
        model,
        val_loaders,
        optimizer=None,
        anomaly_threshold=ANOMALY_THRESHOLD
    )

    val_pointwise_macro_f1 = val_metrics["pointwise_macro_f1"]

    # Update learning rate after validation metric is computed.
    scheduler.step(val_pointwise_macro_f1)

    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "learning_rate": current_lr
    }

    for key, value in train_metrics.items():
        row[f"train_{key}"] = value

    for key, value in val_metrics.items():
        row[f"val_{key}"] = value

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Pointwise Macro F1: {train_metrics['pointwise_macro_f1']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Pointwise Macro F1: {val_pointwise_macro_f1:.4f}"
    )

    if val_pointwise_macro_f1 > best_val_f1 + min_delta:
        best_val_f1 = val_pointwise_macro_f1
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "event_labels": EVENT_LABELS,
                "lengths": LENGTHS,
                "shift_classes": SHIFT_CLASSES,
                "epoch": epoch,
                "best_val_pointwise_macro_f1": best_val_f1,
                "anomaly_threshold": ANOMALY_THRESHOLD
            },
            best_model_path
        )

        print(
            f"New best model saved. "
            f"Best validation pointwise macro F1: {best_val_f1:.4f}"
        )

    else:
        epochs_without_improvement += 1

        print(
            f"No meaningful improvement for {epochs_without_improvement} epoch(s). "
            f"Best validation pointwise macro F1: {best_val_f1:.4f}"
        )

    if epochs_without_improvement >= early_stopping_patience:
        print(f"Early stopping triggered at epoch {epoch}.")
        break


history_df = pd.DataFrame(history)

history_df.to_csv(
    OUTPUT_DIR / "training_history_location_model.csv",
    index=False
)

print("Training finished.")
print("Best validation pointwise macro F1:", best_val_f1)
print("Best model saved to:", best_model_path)

Epoch 01 | LR: 0.001000 | Train Loss: 0.9316 | Train Pointwise Macro F1: 0.0939 | Val Loss: 0.8015 | Val Pointwise Macro F1: 0.0814
New best model saved. Best validation pointwise macro F1: 0.0814
Epoch 02 | LR: 0.001000 | Train Loss: 0.7457 | Train Pointwise Macro F1: 0.1395 | Val Loss: 0.7400 | Val Pointwise Macro F1: 0.1274
New best model saved. Best validation pointwise macro F1: 0.1274
Epoch 03 | LR: 0.001000 | Train Loss: 0.6895 | Train Pointwise Macro F1: 0.1822 | Val Loss: 0.7136 | Val Pointwise Macro F1: 0.1753
New best model saved. Best validation pointwise macro F1: 0.1753
Epoch 04 | LR: 0.001000 | Train Loss: 0.6634 | Train Pointwise Macro F1: 0.2063 | Val Loss: 0.6939 | Val Pointwise Macro F1: 0.2458
New best model saved. Best validation pointwise macro F1: 0.2458
Epoch 05 | LR: 0.001000 | Train Loss: 0.6275 | Train Pointwise Macro F1: 0.2954 | Val Loss: 0.6727 | Val Pointwise Macro F1: 0.2864
New best model saved. Best validation pointwise macro F1: 0.2864
Epoch 06 | LR: 

In [14]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded best model from epoch:", checkpoint["epoch"])
print("Best val pointwise macro F1:", checkpoint["best_val_pointwise_macro_f1"])

Loaded best model from epoch: 50
Best val pointwise macro F1: 0.678286890985991


In [15]:
def extract_persistent_shift_starts(seq, min_persistence):
    """
    Extract shift start indices from a predicted or true shift-regime sequence.

    A shift start is accepted only if the active post-shift region
    continues for at least min_persistence time steps.
    """
    seq = np.asarray(seq)

    starts = []

    diff = np.diff(seq, prepend=0)
    candidate_starts = np.where(diff > 0)[0]

    for start in candidate_starts:
        new_label = seq[start]

        if new_label <= 0:
            continue

        end = start

        while end < len(seq) and seq[end] >= new_label:
            end += 1

        duration = end - start

        if duration >= min_persistence:
            starts.append(start)

    return starts


def extract_starts_from_label_sequence(seq, is_shift, min_persistence=None):
    seq = np.asarray(seq)

    if is_shift:
        if min_persistence is None:
            min_persistence = max(3, int(0.05 * len(seq)))

        starts = extract_persistent_shift_starts(
            seq,
            min_persistence=min_persistence
        )

    else:
        binary = (seq > 0).astype(int)
        diff = np.diff(binary, prepend=0)
        starts = np.where(diff == 1)[0]

    return starts


def match_start_locations(true_starts, pred_starts, tolerance):
    true_starts = list(true_starts)
    pred_starts = list(pred_starts)

    used_pred = set()
    abs_errors = []
    tp = 0

    for true_start in true_starts:
        best_j = None
        best_error = None

        for j, pred_start in enumerate(pred_starts):
            if j in used_pred:
                continue

            error = abs(pred_start - true_start)

            if best_error is None or error < best_error:
                best_error = error
                best_j = j

        if best_error is not None and best_error <= tolerance:
            tp += 1
            used_pred.add(best_j)
            abs_errors.append(best_error)

    fp = len(pred_starts) - len(used_pred)
    fn = len(true_starts) - tp

    return tp, fp, fn, abs_errors

In [16]:
def get_location_predictions_by_length(model, loaders, lengths, anomaly_threshold=0.5):
    """
    Returns predictions separately for each sequence length.

    Output:
        predictions_by_length[length] = {
            "y_true": array with shape (num_samples, length, 5),
            "y_pred": array with shape (num_samples, length, 5)
        }
    """
    model.eval()

    predictions_by_length = {}

    with torch.no_grad():
        for length, loader in zip(lengths, loaders):
            all_true = []
            all_pred = []

            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                shift_logits, anomaly_logits = model(X_batch)

                pred_loc, _ = apply_location_predictions(
                    shift_logits,
                    anomaly_logits,
                    anomaly_threshold=anomaly_threshold
                )
                all_true.append(Yloc_batch.cpu().numpy())
                all_pred.append(pred_loc.cpu().numpy())

            predictions_by_length[length] = {
                "y_true": np.concatenate(all_true, axis=0),
                "y_pred": np.concatenate(all_pred, axis=0)
            }

    return predictions_by_length

In [17]:
def evaluate_event_start_locations_by_length(
    predictions_by_length,
    event_labels,
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.05
):
    """
    Evaluates event-start localization separately for each sequence length.

    For shifts, predicted starts are accepted only if the predicted
    post-shift region persists for at least shift_persistence_ratio
    of the sequence length.

    TN definition:
        sample-level true negative for a given event type:
        no true start exists and no predicted start exists.
    """
    rows = []

    for length, pred_dict in predictions_by_length.items():
        y_true_loc = pred_dict["y_true"]
        y_pred_loc = pred_dict["y_pred"]

        n_samples, seq_len, n_events = y_true_loc.shape

        tolerance = max(3, int(seq_len * tolerance_ratio))
        min_persistence = max(3, int(seq_len * shift_persistence_ratio))

        for event_idx, label in enumerate(event_labels):
            is_shift = event_idx < 3

            total_tp = 0
            total_fp = 0
            total_fn = 0
            total_tn = 0
            all_errors = []

            for sample_idx in range(n_samples):
                true_seq = y_true_loc[sample_idx, :, event_idx]
                pred_seq = y_pred_loc[sample_idx, :, event_idx]

                true_starts = extract_starts_from_label_sequence(
                    true_seq,
                    is_shift=is_shift,
                    min_persistence=1 if is_shift else None
)

                pred_starts = extract_starts_from_label_sequence(
                    pred_seq,
                    is_shift=is_shift,
                    min_persistence=min_persistence if is_shift else None
                )

                # Sample-level TN: no real start and no predicted start.
                if len(true_starts) == 0 and len(pred_starts) == 0:
                    total_tn += 1
                    continue

                tp, fp, fn, errors = match_start_locations(
                    true_starts=true_starts,
                    pred_starts=pred_starts,
                    tolerance=tolerance
                )

                total_tp += tp
                total_fp += fp
                total_fn += fn
                all_errors.extend(errors)

            precision = total_tp / (total_tp + total_fp + 1e-8)
            recall = total_tp / (total_tp + total_fn + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)

            specificity = total_tn / (total_tn + total_fp + 1e-8)
            balanced_accuracy = 0.5 * (recall + specificity)

            mean_abs_error = np.mean(all_errors) if len(all_errors) > 0 else np.nan

            rows.append({
                "length": length,
                "event": label,
                "tolerance_points": tolerance,
                "shift_min_persistence": min_persistence if is_shift else np.nan,
                "tp": total_tp,
                "fp": total_fp,
                "fn": total_fn,
                "tn": total_tn,
                "start_precision": precision,
                "start_recall": recall,
                "start_f1": f1,
                "start_specificity": specificity,
                "start_balanced_accuracy": balanced_accuracy,
                "mean_abs_start_error": mean_abs_error
            })

    return pd.DataFrame(rows)

In [18]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    multilabel_confusion_matrix
)

def apply_location_predictions(shift_logits, anomaly_logits, anomaly_threshold=0.5):
    """
    Shift outputs:
        argmax over regime classes.

    Anomaly outputs:
        sigmoid + threshold.

    anomaly_threshold can be:
        - scalar, e.g. 0.5
        - array/list of length 2, e.g. [0.93, 0.95]
    """

    shift_pred = torch.argmax(shift_logits, dim=-1)

    anomaly_prob = torch.sigmoid(anomaly_logits)

    if np.isscalar(anomaly_threshold):
        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()
    else:
        anomaly_threshold = torch.tensor(
            anomaly_threshold,
            dtype=anomaly_prob.dtype,
            device=anomaly_prob.device
        ).view(1, 1, -1)

        anomaly_pred = (anomaly_prob >= anomaly_threshold).long()

    pred_loc = torch.cat([shift_pred, anomaly_pred], dim=-1)

    return pred_loc, anomaly_prob


def collect_anomaly_probs_and_labels(model, loaders):
    model.eval()

    all_true = []
    all_prob = []

    with torch.no_grad():
        for loader in loaders:
            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                _, anomaly_logits = model(X_batch)
                anomaly_prob = torch.sigmoid(anomaly_logits)

                all_true.append(
                    Yloc_batch[:, :, 3:5].cpu().numpy().reshape(-1, 2)
                )

                all_prob.append(
                    anomaly_prob.cpu().numpy().reshape(-1, 2)
                )

    y_true = np.concatenate(all_true, axis=0)
    y_prob = np.concatenate(all_prob, axis=0)

    return y_true, y_prob


def tune_anomaly_thresholds_per_class_location(
    model,
    val_loaders,
    thresholds=None
):
    """
    Tunes only anomaly thresholds using validation set.

    Tuned classes:
        point_anomaly
        collective_anomaly

    Shift classes are not threshold-tuned because their location predictions
    use argmax over shift regimes.
    """

    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.01)

    y_true, y_prob = collect_anomaly_probs_and_labels(
        model=model,
        loaders=val_loaders
    )

    rows = []
    best_thresholds = []

    for anomaly_idx, label in enumerate(EVENT_LABELS[3:5]):
        best_threshold = 0.5
        best_f1 = -1
        best_precision = 0
        best_recall = 0

        for threshold in thresholds:
            y_pred = (y_prob[:, anomaly_idx] >= threshold).astype(int)

            precision = precision_score(
                y_true[:, anomaly_idx],
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_true[:, anomaly_idx],
                y_pred,
                zero_division=0
            )

            f1 = f1_score(
                y_true[:, anomaly_idx],
                y_pred,
                zero_division=0
            )

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                best_precision = precision
                best_recall = recall

        best_thresholds.append(best_threshold)

        rows.append({
            "label": label,
            "best_threshold": best_threshold,
            "validation_precision": best_precision,
            "validation_recall": best_recall,
            "validation_f1": best_f1
        })

    threshold_df = pd.DataFrame(rows)

    return np.array(best_thresholds), threshold_df


TUNED_ANOMALY_THRESHOLDS, location_threshold_df = tune_anomaly_thresholds_per_class_location(
    model=model,
    val_loaders=val_loaders,
    thresholds=np.arange(0.05, 0.96, 0.01)
)

location_threshold_df.to_csv(
    OUTPUT_DIR / "validation_threshold_tuning_location_anomalies.csv",
    index=False
)

print("Tuned anomaly thresholds:", TUNED_ANOMALY_THRESHOLDS)

location_threshold_df

Tuned anomaly thresholds: [0.92 0.94]


,label,best_threshold,validation_precision,validation_recall,validation_f1
0,point_anomaly,0.92,0.522523,0.763158,0.620321
1,collective_anomaly,0.94,0.962307,0.895772,0.927848


In [19]:
location_dataset_specs = [
    ("validation", val_loaders),
    ("ood_params", ood_params_loaders),
    ("ood_background", ood_background_loaders)
]


def get_location_flat_predictions(model, loaders, anomaly_threshold=0.5):
    """
    Returns flattened pointwise true and predicted labels.

    Output:
        y_true_flat: (total_time_points, 5)
        y_pred_flat: (total_time_points, 5)
    """

    model.eval()

    all_true_flat = []
    all_pred_flat = []

    with torch.no_grad():
        for loader in loaders:
            for X_batch, Yloc_batch in loader:
                X_batch = X_batch.to(device)

                shift_logits, anomaly_logits = model(X_batch)

                pred_loc, _ = apply_location_predictions(
                    shift_logits,
                    anomaly_logits,
                    anomaly_threshold=anomaly_threshold
                )

                true_flat = Yloc_batch.cpu().numpy().reshape(-1, len(EVENT_LABELS))
                pred_flat = pred_loc.cpu().numpy().reshape(-1, len(EVENT_LABELS))

                all_true_flat.append(true_flat)
                all_pred_flat.append(pred_flat)

    y_true_flat = np.concatenate(all_true_flat, axis=0)
    y_pred_flat = np.concatenate(all_pred_flat, axis=0)

    return y_true_flat, y_pred_flat


def compute_classwise_pointwise_location_metrics(
    y_true_flat,
    y_pred_flat,
    label_names,
    dataset_name,
    setting,
    anomaly_threshold
):
    """
    Class-wise pointwise metrics.

    For shift classes:
        regime labels are converted to active/not-active using > 0.

    For anomaly classes:
        labels are already binary.
    """

    y_true_binary = y_true_flat.copy()
    y_pred_binary = y_pred_flat.copy()

    y_true_binary[:, :3] = (y_true_binary[:, :3] > 0).astype(int)
    y_pred_binary[:, :3] = (y_pred_binary[:, :3] > 0).astype(int)

    ml_cm = multilabel_confusion_matrix(y_true_binary, y_pred_binary)

    rows = []

    anomaly_threshold = np.asarray(anomaly_threshold)

    for i, label in enumerate(label_names):
        tn, fp, fn, tp = ml_cm[i].ravel()

        if i < 3:
            decision_rule = "argmax_regime"
            threshold_value = np.nan
        else:
            decision_rule = "sigmoid_threshold"

            if anomaly_threshold.ndim == 0:
                threshold_value = float(anomaly_threshold)
            else:
                threshold_value = float(anomaly_threshold[i - 3])

        rows.append({
            "dataset": dataset_name,
            "setting": setting,
            "label": label,
            "decision_rule": decision_rule,
            "threshold": threshold_value,
            "precision": precision_score(
                y_true_binary[:, i],
                y_pred_binary[:, i],
                zero_division=0
            ),
            "recall": recall_score(
                y_true_binary[:, i],
                y_pred_binary[:, i],
                zero_division=0
            ),
            "f1": f1_score(
                y_true_binary[:, i],
                y_pred_binary[:, i],
                zero_division=0
            ),
            "support": int(y_true_binary[:, i].sum()),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp)
        })

    return pd.DataFrame(rows)


def evaluate_pointwise_location_for_all_datasets(
    model,
    dataset_specs,
    anomaly_threshold,
    setting_name,
    save_suffix
):
    overall_rows = []
    classwise_rows = []

    for dataset_name, loaders in dataset_specs:
        metrics = run_one_epoch_location(
            model,
            loaders,
            optimizer=None,
            anomaly_threshold=anomaly_threshold
        )

        if np.isscalar(anomaly_threshold):
            point_threshold = float(anomaly_threshold)
            collective_threshold = float(anomaly_threshold)
        else:
            point_threshold = float(anomaly_threshold[0])
            collective_threshold = float(anomaly_threshold[1])

        overall_rows.append({
            "dataset": dataset_name,
            "setting": setting_name,
            "point_anomaly_threshold": point_threshold,
            "collective_anomaly_threshold": collective_threshold,
            **metrics
        })

        y_true_flat, y_pred_flat = get_location_flat_predictions(
            model=model,
            loaders=loaders,
            anomaly_threshold=anomaly_threshold
        )

        classwise_df = compute_classwise_pointwise_location_metrics(
            y_true_flat=y_true_flat,
            y_pred_flat=y_pred_flat,
            label_names=EVENT_LABELS,
            dataset_name=dataset_name,
            setting=setting_name,
            anomaly_threshold=anomaly_threshold
        )

        classwise_rows.append(classwise_df)

    overall_df = pd.DataFrame(overall_rows)

    classwise_df = pd.concat(
        classwise_rows,
        ignore_index=True
    )

    overall_df.to_csv(
        OUTPUT_DIR / f"location_pointwise_overall_{save_suffix}.csv",
        index=False
    )

    classwise_df.to_csv(
        OUTPUT_DIR / f"location_pointwise_classwise_{save_suffix}.csv",
        index=False
    )

    return overall_df, classwise_df


pointwise_overall_default, pointwise_classwise_default = evaluate_pointwise_location_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    anomaly_threshold=0.5,
    setting_name="default_0.5",
    save_suffix="default_threshold"
)

pointwise_overall_tuned, pointwise_classwise_tuned = evaluate_pointwise_location_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    anomaly_threshold=TUNED_ANOMALY_THRESHOLDS,
    setting_name="tuned",
    save_suffix="tuned_threshold"
)

display(pointwise_overall_default)
display(pointwise_classwise_default)

display(pointwise_overall_tuned)
display(pointwise_classwise_tuned)

,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,loss,shift_loss,anomaly_loss,pointwise_macro_f1,pointwise_micro_f1,pointwise_macro_precision,pointwise_macro_recall,pointwise_hamming_loss,mean_shift_regime_accuracy,variance_shift_regime_accuracy,trend_shift_regime_accuracy
0,validation,default_0.5,0.5,0.5,0.473651,0.304019,0.169633,0.678287,0.735468,0.725118,0.744592,0.051122,0.904715,0.913956,0.870632
1,ood_params,default_0.5,0.5,0.5,0.683667,0.414456,0.269211,0.530996,0.593870,0.640967,0.479185,0.071597,0.889961,0.867138,0.846445
2,ood_background,default_0.5,0.5,0.5,0.947641,0.522807,0.424834,0.494884,0.467793,0.505977,0.580656,0.104486,0.782405,0.842621,0.826638


,dataset,setting,label,decision_rule,threshold,precision,recall,f1,support,tn,fp,fn,tp
0,validation,default_0.5,mean_shift,argmax_regime,NaN,0.858127,0.690810,0.765431,132763,620074,15163,41049,91714
1,validation,default_0.5,variance_shift,argmax_regime,NaN,0.908488,0.742858,0.817367,138865,618744,10391,35708,103157
2,validation,default_0.5,trend_shift,argmax_regime,NaN,0.805099,0.435016,0.564837,130956,623253,13791,73988,56968
3,validation,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.227496,0.914474,0.364351,152,767376,472,13,139
4,validation,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.826381,0.939800,0.879448,22259,741346,4395,1340,20919
5,ood_params,default_0.5,mean_shift,argmax_regime,NaN,0.754056,0.601016,0.668894,124742,618805,24453,49770,74972
6,ood_params,default_0.5,variance_shift,argmax_regime,NaN,0.818714,0.421603,0.556587,130706,625092,12202,75600,55106
7,ood_params,default_0.5,trend_shift,argmax_regime,NaN,0.696272,0.331773,0.449405,132015,616879,19106,88216,43799
8,ood_params,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.050847,0.098684,0.067114,152,767568,280,137,15
9,ood_params,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.884948,0.942847,0.912981,28765,735709,3526,1644,27121


,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,loss,shift_loss,anomaly_loss,pointwise_macro_f1,pointwise_micro_f1,pointwise_macro_precision,pointwise_macro_recall,pointwise_hamming_loss,mean_shift_regime_accuracy,variance_shift_regime_accuracy,trend_shift_regime_accuracy
0,validation,tuned,0.92,0.94,0.473651,0.304019,0.169633,0.739161,0.737719,0.811309,0.705523,0.050347,0.904715,0.913956,0.870632
1,ood_params,tuned,0.92,0.94,0.683667,0.414456,0.269211,0.527426,0.592533,0.664670,0.451021,0.071286,0.889961,0.867138,0.846445
2,ood_background,tuned,0.92,0.94,0.947641,0.522807,0.424834,0.534396,0.468395,0.582961,0.520983,0.103240,0.782405,0.842621,0.826638


,dataset,setting,label,decision_rule,threshold,precision,recall,f1,support,tn,fp,fn,tp
0,validation,tuned,mean_shift,argmax_regime,NaN,0.858127,0.690810,0.765431,132763,620074,15163,41049,91714
1,validation,tuned,variance_shift,argmax_regime,NaN,0.908488,0.742858,0.817367,138865,618744,10391,35708,103157
2,validation,tuned,trend_shift,argmax_regime,NaN,0.805099,0.435016,0.564837,130956,623253,13791,73988,56968
3,validation,tuned,point_anomaly,sigmoid_threshold,0.92,0.522523,0.763158,0.620321,152,767742,106,36,116
4,validation,tuned,collective_anomaly,sigmoid_threshold,0.94,0.962307,0.895772,0.927848,22259,744960,781,2320,19939
5,ood_params,tuned,mean_shift,argmax_regime,NaN,0.754056,0.601016,0.668894,124742,618805,24453,49770,74972
6,ood_params,tuned,variance_shift,argmax_regime,NaN,0.818714,0.421603,0.556587,130706,625092,12202,75600,55106
7,ood_params,tuned,trend_shift,argmax_regime,NaN,0.696272,0.331773,0.449405,132015,616879,19106,88216,43799
8,ood_params,tuned,point_anomaly,sigmoid_threshold,0.92,0.076923,0.026316,0.039216,152,767800,48,148,4
9,ood_params,tuned,collective_anomaly,sigmoid_threshold,0.94,0.977384,0.874396,0.923026,28765,738653,582,3613,25152


In [20]:
def evaluate_start_locations_for_all_datasets(
    model,
    dataset_specs,
    lengths,
    anomaly_threshold,
    setting_name,
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1,
    save_suffix="default_threshold"
):
    start_classwise_rows = []

    for dataset_name, loaders in dataset_specs:
        predictions_by_length = get_location_predictions_by_length(
            model=model,
            loaders=loaders,
            lengths=lengths,
            anomaly_threshold=anomaly_threshold
        )

        start_df = evaluate_event_start_locations_by_length(
            predictions_by_length=predictions_by_length,
            event_labels=EVENT_LABELS,
            tolerance_ratio=tolerance_ratio,
            shift_persistence_ratio=shift_persistence_ratio
        )

        start_df["dataset"] = dataset_name
        start_df["setting"] = setting_name

        if np.isscalar(anomaly_threshold):
            start_df["point_anomaly_threshold"] = float(anomaly_threshold)
            start_df["collective_anomaly_threshold"] = float(anomaly_threshold)
        else:
            start_df["point_anomaly_threshold"] = float(anomaly_threshold[0])
            start_df["collective_anomaly_threshold"] = float(anomaly_threshold[1])

        start_classwise_rows.append(start_df)

    start_classwise_df = pd.concat(
        start_classwise_rows,
        ignore_index=True
    )

    overall_rows = []

    for (dataset_name, setting_name), group in start_classwise_df.groupby(["dataset", "setting"]):
        total_tp = group["tp"].sum()
        total_fp = group["fp"].sum()
        total_fn = group["fn"].sum()
        total_tn = group["tn"].sum()

        start_precision = total_tp / (total_tp + total_fp + 1e-8)
        start_recall = total_tp / (total_tp + total_fn + 1e-8)
        start_f1 = 2 * start_precision * start_recall / (start_precision + start_recall + 1e-8)

        start_specificity = total_tn / (total_tn + total_fp + 1e-8)
        start_balanced_accuracy = 0.5 * (start_recall + start_specificity)

        valid_error_rows = group.dropna(subset=["mean_abs_start_error"])

        if len(valid_error_rows) > 0:
            weighted_error = np.average(
                valid_error_rows["mean_abs_start_error"],
                weights=valid_error_rows["tp"].clip(lower=1)
            )
        else:
            weighted_error = np.nan

        overall_rows.append({
            "dataset": dataset_name,
            "setting": setting_name,
            "point_anomaly_threshold": group["point_anomaly_threshold"].iloc[0],
            "collective_anomaly_threshold": group["collective_anomaly_threshold"].iloc[0],
            "tp": total_tp,
            "fp": total_fp,
            "fn": total_fn,
            "tn": total_tn,
            "start_precision": start_precision,
            "start_recall": start_recall,
            "start_f1": start_f1,
            "start_specificity": start_specificity,
            "start_balanced_accuracy": start_balanced_accuracy,
            "mean_abs_start_error": weighted_error
        })

    start_overall_df = pd.DataFrame(overall_rows)

    start_overall_df.to_csv(
        OUTPUT_DIR / f"location_start_overall_{save_suffix}.csv",
        index=False
    )

    start_classwise_df.to_csv(
        OUTPUT_DIR / f"location_start_classwise_by_length_{save_suffix}.csv",
        index=False
    )

    return start_overall_df, start_classwise_df


start_overall_default, start_classwise_default = evaluate_start_locations_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    lengths=LENGTHS,
    anomaly_threshold=0.5,
    setting_name="default_0.5",
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1,
    save_suffix="default_threshold"
)

start_overall_tuned, start_classwise_tuned = evaluate_start_locations_for_all_datasets(
    model=model,
    dataset_specs=location_dataset_specs,
    lengths=LENGTHS,
    anomaly_threshold=TUNED_ANOMALY_THRESHOLDS,
    setting_name="tuned",
    tolerance_ratio=0.05,
    shift_persistence_ratio=0.1,
    save_suffix="tuned_threshold"
)

display(start_overall_default)
display(start_classwise_default)

display(start_overall_tuned)
display(start_classwise_tuned)

,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error
0,ood_background,default_0.5,0.5,0.5,553,1587,1016,2758,0.258411,0.352454,0.298194,0.634753,0.493603,6.887884
1,ood_params,default_0.5,0.5,0.5,584,973,1032,3005,0.375080,0.361386,0.368106,0.755405,0.558395,7.481164
2,validation,default_0.5,0.5,0.5,811,1174,776,2971,0.408564,0.511027,0.454087,0.716767,0.613897,6.594328


,length,event,tolerance_points,shift_min_persistence,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold
0,400,mean_shift,20,40.0,43,28,83,213,0.605634,0.341270,0.436548,0.883817,0.612544,5.930233,validation,default_0.5,0.5,0.5
1,400,variance_shift,20,40.0,48,35,79,211,0.578313,0.377953,0.457143,0.857724,0.617838,6.458333,validation,default_0.5,0.5,0.5
2,400,trend_shift,20,40.0,16,25,104,217,0.390244,0.133333,0.198758,0.896694,0.515014,9.187500,validation,default_0.5,0.5,0.5
3,400,point_anomaly,20,NaN,45,97,6,201,0.316901,0.882353,0.466321,0.674497,0.778425,0.000000,validation,default_0.5,0.5,0.5
4,400,collective_anomaly,20,NaN,88,36,12,197,0.709677,0.880000,0.785714,0.845494,0.862747,1.875000,validation,default_0.5,0.5,0.5
5,800,mean_shift,40,80.0,57,28,73,209,0.670588,0.438462,0.530233,0.881857,0.660159,7.894737,validation,default_0.5,0.5,0.5
6,800,variance_shift,40,80.0,57,30,72,217,0.655172,0.441860,0.527778,0.878543,0.660201,12.017544,validation,default_0.5,0.5,0.5
7,800,trend_shift,40,80.0,18,57,108,206,0.240000,0.142857,0.179104,0.783270,0.463064,19.333333,validation,default_0.5,0.5,0.5
8,800,point_anomaly,40,NaN,48,151,4,179,0.241206,0.923077,0.382470,0.542424,0.732751,0.000000,validation,default_0.5,0.5,0.5
9,800,collective_anomaly,40,NaN,98,133,2,166,0.424242,0.980000,0.592145,0.555184,0.767592,2.459184,validation,default_0.5,0.5,0.5


,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error
0,ood_background,tuned,0.92,0.94,498,802,1071,3037,0.383077,0.317400,0.347159,0.791091,0.554246,7.343373
1,ood_params,tuned,0.92,0.94,553,474,1063,3273,0.538462,0.342203,0.418464,0.873499,0.607851,8.613020
2,validation,tuned,0.92,0.94,773,483,814,3275,0.615446,0.487083,0.543792,0.871474,0.679278,6.567917


,length,event,tolerance_points,shift_min_persistence,tp,fp,fn,tn,start_precision,start_recall,start_f1,start_specificity,start_balanced_accuracy,mean_abs_start_error,dataset,setting,point_anomaly_threshold,collective_anomaly_threshold
0,400,mean_shift,20,40.0,43,28,83,213,0.605634,0.341270,0.436548,0.883817,0.612544,5.930233,validation,tuned,0.92,0.94
1,400,variance_shift,20,40.0,48,35,79,211,0.578313,0.377953,0.457143,0.857724,0.617838,6.458333,validation,tuned,0.92,0.94
2,400,trend_shift,20,40.0,16,25,104,217,0.390244,0.133333,0.198758,0.896694,0.515014,9.187500,validation,tuned,0.92,0.94
3,400,point_anomaly,20,NaN,34,21,17,250,0.618182,0.666667,0.641509,0.922509,0.794588,0.000000,validation,tuned,0.92,0.94
4,400,collective_anomaly,20,NaN,79,10,21,218,0.887640,0.790000,0.835979,0.956140,0.873070,1.784810,validation,tuned,0.92,0.94
5,800,mean_shift,40,80.0,57,28,73,209,0.670588,0.438462,0.530233,0.881857,0.660159,7.894737,validation,tuned,0.92,0.94
6,800,variance_shift,40,80.0,57,30,72,217,0.655172,0.441860,0.527778,0.878543,0.660201,12.017544,validation,tuned,0.92,0.94
7,800,trend_shift,40,80.0,18,57,108,206,0.240000,0.142857,0.179104,0.783270,0.463064,19.333333,validation,tuned,0.92,0.94
8,800,point_anomaly,40,NaN,45,33,7,240,0.576923,0.865385,0.692308,0.879121,0.872253,0.000000,validation,tuned,0.92,0.94
9,800,collective_anomaly,40,NaN,95,14,5,218,0.871560,0.950000,0.909091,0.939655,0.944828,2.157895,validation,tuned,0.92,0.94


In [21]:
def make_wide_comparison(default_df, tuned_df, merge_keys, output_name):
    comparison_wide = default_df.merge(
        tuned_df,
        on=merge_keys,
        suffixes=("_default", "_tuned")
    )

    for col in default_df.columns:
        if col in merge_keys:
            continue

        if pd.api.types.is_numeric_dtype(default_df[col]):
            comparison_wide[f"{col}_change"] = (
                comparison_wide[f"{col}_tuned"] -
                comparison_wide[f"{col}_default"]
            )

    comparison_wide.to_csv(
        OUTPUT_DIR / output_name,
        index=False
    )

    return comparison_wide


# A1. Overall pointwise comparison
pointwise_overall_comparison = make_wide_comparison(
    default_df=pointwise_overall_default,
    tuned_df=pointwise_overall_tuned,
    merge_keys=["dataset"],
    output_name="comparison_pointwise_overall_default_vs_tuned.csv"
)

# A2. Class-wise pointwise comparison
pointwise_classwise_comparison = make_wide_comparison(
    default_df=pointwise_classwise_default,
    tuned_df=pointwise_classwise_tuned,
    merge_keys=["dataset", "label"],
    output_name="comparison_pointwise_classwise_default_vs_tuned.csv"
)

# B1. Overall event-start comparison
start_overall_comparison = make_wide_comparison(
    default_df=start_overall_default,
    tuned_df=start_overall_tuned,
    merge_keys=["dataset"],
    output_name="comparison_start_overall_default_vs_tuned.csv"
)

# B2. Class-wise/event-wise event-start comparison
start_classwise_comparison = make_wide_comparison(
    default_df=start_classwise_default,
    tuned_df=start_classwise_tuned,
    merge_keys=["dataset", "length", "event"],
    output_name="comparison_start_classwise_by_length_default_vs_tuned.csv"
)

display(pointwise_overall_comparison)
display(pointwise_classwise_comparison)
display(start_overall_comparison)
display(start_classwise_comparison)

,dataset,setting_default,point_anomaly_threshold_default,collective_anomaly_threshold_default,loss_default,shift_loss_default,anomaly_loss_default,pointwise_macro_f1_default,pointwise_micro_f1_default,pointwise_macro_precision_default,...,shift_loss_change,anomaly_loss_change,pointwise_macro_f1_change,pointwise_micro_f1_change,pointwise_macro_precision_change,pointwise_macro_recall_change,pointwise_hamming_loss_change,mean_shift_regime_accuracy_change,variance_shift_regime_accuracy_change,trend_shift_regime_accuracy_change
0,validation,default_0.5,0.5,0.5,0.473651,0.304019,0.169633,0.678287,0.735468,0.725118,...,0.0,0.0,0.060874,0.002250,0.086191,-0.039069,-0.000775,0.0,0.0,0.0
1,ood_params,default_0.5,0.5,0.5,0.683667,0.414456,0.269211,0.530996,0.593870,0.640967,...,0.0,0.0,-0.003571,-0.001337,0.023702,-0.028164,-0.000311,0.0,0.0,0.0
2,ood_background,default_0.5,0.5,0.5,0.947641,0.522807,0.424834,0.494884,0.467793,0.505977,...,0.0,0.0,0.039512,0.000602,0.076984,-0.059673,-0.001246,0.0,0.0,0.0


,dataset,setting_default,label,decision_rule_default,threshold_default,precision_default,recall_default,f1_default,support_default,tn_default,...,tp_tuned,threshold_change,precision_change,recall_change,f1_change,support_change,tn_change,fp_change,fn_change,tp_change
0,validation,default_0.5,mean_shift,argmax_regime,NaN,0.858127,0.690810,0.765431,132763,620074,...,91714,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
1,validation,default_0.5,variance_shift,argmax_regime,NaN,0.908488,0.742858,0.817367,138865,618744,...,103157,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
2,validation,default_0.5,trend_shift,argmax_regime,NaN,0.805099,0.435016,0.564837,130956,623253,...,56968,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
3,validation,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.227496,0.914474,0.364351,152,767376,...,116,0.42,0.295027,-0.151316,0.255970,0,366,-366,23,-23
4,validation,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.826381,0.939800,0.879448,22259,741346,...,19939,0.44,0.135926,-0.044027,0.048400,0,3614,-3614,980,-980
5,ood_params,default_0.5,mean_shift,argmax_regime,NaN,0.754056,0.601016,0.668894,124742,618805,...,74972,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
6,ood_params,default_0.5,variance_shift,argmax_regime,NaN,0.818714,0.421603,0.556587,130706,625092,...,55106,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
7,ood_params,default_0.5,trend_shift,argmax_regime,NaN,0.696272,0.331773,0.449405,132015,616879,...,43799,NaN,0.000000,0.000000,0.000000,0,0,0,0,0
8,ood_params,default_0.5,point_anomaly,sigmoid_threshold,0.5,0.050847,0.098684,0.067114,152,767568,...,4,0.42,0.026076,-0.072368,-0.027898,0,232,-232,11,-11
9,ood_params,default_0.5,collective_anomaly,sigmoid_threshold,0.5,0.884948,0.942847,0.912981,28765,735709,...,25152,0.44,0.092436,-0.068451,0.010046,0,2944,-2944,1969,-1969


,dataset,setting_default,point_anomaly_threshold_default,collective_anomaly_threshold_default,tp_default,fp_default,fn_default,tn_default,start_precision_default,start_recall_default,...,tp_change,fp_change,fn_change,tn_change,start_precision_change,start_recall_change,start_f1_change,start_specificity_change,start_balanced_accuracy_change,mean_abs_start_error_change
0,ood_background,default_0.5,0.5,0.5,553,1587,1016,2758,0.258411,0.352454,...,-55,-785,55,279,0.124666,-0.035054,0.048966,0.156339,0.060642,0.455489
1,ood_params,default_0.5,0.5,0.5,584,973,1032,3005,0.375080,0.361386,...,-31,-499,31,268,0.163381,-0.019183,0.050358,0.118094,0.049455,1.131856
2,validation,default_0.5,0.5,0.5,811,1174,776,2971,0.408564,0.511027,...,-38,-691,38,304,0.206882,-0.023945,0.089704,0.154707,0.065381,-0.026411


,length,event,tolerance_points_default,shift_min_persistence_default,tp_default,fp_default,fn_default,tn_default,start_precision_default,start_recall_default,...,fn_change,tn_change,start_precision_change,start_recall_change,start_f1_change,start_specificity_change,start_balanced_accuracy_change,mean_abs_start_error_change,point_anomaly_threshold_change,collective_anomaly_threshold_change
0,400,mean_shift,20,40.0,43,28,83,213,0.605634,0.341270,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.42,0.44
1,400,variance_shift,20,40.0,48,35,79,211,0.578313,0.377953,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.42,0.44
2,400,trend_shift,20,40.0,16,25,104,217,0.390244,0.133333,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.42,0.44
3,400,point_anomaly,20,NaN,45,97,6,201,0.316901,0.882353,...,11,49,0.301280,-0.215686,0.175188,0.248013,0.016163,0.000000,0.42,0.44
4,400,collective_anomaly,20,NaN,88,36,12,197,0.709677,0.880000,...,9,21,0.177963,-0.090000,0.050265,0.110647,0.010323,-0.090190,0.42,0.44
5,800,mean_shift,40,80.0,57,28,73,209,0.670588,0.438462,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.42,0.44
6,800,variance_shift,40,80.0,57,30,72,217,0.655172,0.441860,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.42,0.44
7,800,trend_shift,40,80.0,18,57,108,206,0.240000,0.142857,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.42,0.44
8,800,point_anomaly,40,NaN,48,151,4,179,0.241206,0.923077,...,3,61,0.335717,-0.057692,0.309838,0.336697,0.139502,0.000000,0.42,0.44
9,800,collective_anomaly,40,NaN,98,133,2,166,0.424242,0.980000,...,3,52,0.447317,-0.030000,0.316946,0.384471,0.177236,-0.301289,0.42,0.44
